# GOES–MRMS: explore a real forecast
Companion demo for *Spatially Explicit Deep Learning Nowcasting of Convection Initiation Using GOES and MRMS*.

This notebook loads the released model and a real 64 × 64 TEST crop. It predicts raw CI probabilities for **0–30** and **30–60 min**, then lets you explore the inputs and outputs. CPU is sufficient. The published checkpoint download is approximately **309 MiB**.

The repository and its Release must be public for the automatic Colab setup. Locally, open this notebook from the source checkout.

## 1 · Locate the code and install demo dependencies

In [ ]:
from pathlib import Path
import subprocess, sys
REPOSITORY = 'https://github.com/Marccrampe/goes-mrms-ci-nowcasting'
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'models/resunet.py').is_file()), None)
if root is None:
    checkout = Path.cwd() / 'goes-mrms-ci-nowcasting'
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, str(checkout)], check=True)
    root = checkout.resolve()
assert (root / 'examples/mini_case.npz').is_file(), 'The real miniature sample is missing from the checkout.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements-demo.txt')], check=True)
sys.path.insert(0, str(root / 'tools'))
print('Source checkout:', root)

## 2 · Verify the weights and run actual inference
The helper downloads the named asset from the latest published Release if needed, verifies its SHA-256 and calls the repository's inference program. The two windows are predicted by the same released model.

In [ ]:
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass
from demo_support import run_inference, show_demo
output = run_inference(root)
import numpy as np
with np.load(output) as data:
    probability = data['probability']
assert probability.shape == (1, 2, 64, 64)
assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
print('Forecast shape:', probability.shape)
show_demo(root)

## 3 · Explore the example
Select an input-history frame and a forecast window. The colour-scale control changes the displayed range only: **it does not change forecast values, thresholds or verification**. Predictions remain fixed at the same initialization time even when you browse the earlier input frames.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
window = widgets.Dropdown(options=[('0–30 min', 0), ('30–60 min', 1)], description='Forecast:')
history = widgets.Dropdown(options=[('−30 min', 0), ('−20 min', 1), ('−10 min', 2), ('0 min', 3)], value=3, description='Input time:')
scale = widgets.Dropdown(options=[0.1, 0.25, 0.5, 1.0], value=0.5, description='Colour max:')
view = widgets.interactive_output(lambda window, history, display_max: show_demo(root, window, history, display_max),
    {'window': window, 'history': history, 'display_max': scale})
display(widgets.HBox([window, history, scale]), view)

## What this demonstrates
- Real physical GOES/MRMS inputs, original normalization and the released checkpoint.
- Two raw probability fields from CPU inference.
- Interactive display without modifying the model or scientific scores.

This miniature is not a full-domain verification case. It does not demonstrate the complete catalogue/preprocessing/GPU-training reproduction, and it does not apply isotonic calibration. See `docs/pipeline.md` and `docs/validation.md` for the full workflow and validation scope.